Predicting Credit Card Approvals Project

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV

In [2]:
# Load the dataset
cc_apps = pd.read_csv("cc_approvals.data", header=None)

In [3]:
# Replace ? with NaN
cc_apps_nans_replaced = cc_apps.replace("?", np.nan)

# Copying the dataframe
cc_apps_imputed = cc_apps_nans_replaced.copy()

In [4]:
# Fill missing values
for col in cc_apps_imputed.columns:
    if pd.api.types.is_numeric_dtype(cc_apps_imputed[col]):
        # Mean for numeric
        cc_apps_imputed[col] = cc_apps_imputed[col].fillna(cc_apps_imputed[col].mean())
    else:
        # Most frequent for categorical
        cc_apps_imputed[col] = cc_apps_imputed[col].fillna(
            cc_apps_imputed[col].value_counts().index[0]
        )

In [5]:
# One-hot encode
cc_apps_encoded = pd.get_dummies(cc_apps_imputed, drop_first=True)

In [6]:
# Define features and target
X = cc_apps_encoded.iloc[:, :-1].values
y = cc_apps_encoded.iloc[:, -1].values

# Split train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=1234)

In [7]:
# Scale the data
scaler = StandardScaler()
rescaledX_train = scaler.fit_transform(X_train)
rescaledX_test = scaler.transform(X_test)

In [8]:
# Train logistic regression
logreg = LogisticRegression()
logreg.fit(rescaledX_train, y_train)
 
# Predict on training set
y_train_pred = logreg.predict(rescaledX_train)
 

In [9]:
# Print confusion matrix
print(confusion_matrix(y_train, y_train_pred))

[[200   2]
 [  2 258]]


In [10]:
# Define grid search params
tol = [0.01, 0.001, 0.0001]
max_iter = [100, 150, 200]
param_grid = dict(tol=tol, max_iter=max_iter)
 
# Run grid search CV
grid_model = GridSearchCV(estimator=logreg, param_grid=param_grid, cv=5)
grid_model_result = grid_model.fit(rescaledX_train, y_train)
 
# Print best params
best_train_score, best_train_params = grid_model_result.best_score_, grid_model_result.best_params_
print("Best: %f using %s" % (best_train_score, best_train_params))

Best: 0.822487 using {'max_iter': 100, 'tol': 0.01}


In [11]:
# Score on test set
best_model = grid_model_result.best_estimator_
best_score = best_model.score(rescaledX_test, y_test)
print("Accuracy of logistic regression classifier: ", best_score)

Accuracy of logistic regression classifier:  0.7982456140350878
